# 05 · Per-Benchmark Breakdown

**Purpose:** show the **mean value per (benchmark × language)** as small-multiples bar
charts — one panel per CLBG benchmark, 18 language bars each, sorted most-efficient
first and coloured by execution paradigm. Complements the aggregate rankings in
notebooks 02–04 by exposing per-benchmark behaviour.

**Three figures** (each = 8 benchmark panels):
1. CPU Energy (J)
2. Memory / DRAM Energy (J)
3. Execution Time (ms)

Values are the per-(language × benchmark) cell means from `results_clean_runs.csv`
(`plot_style.cell_means`). Each panel has its own x-scale, so bars show the ranking
**within** that benchmark.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import plot_style as ps
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
PARADIGM        = ps.PARADIGM
PARADIGM_COLORS = ps.PARADIGM_COLORS
PARADIGM_ORDER  = ps.PARADIGM_ORDER

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)
print(f"Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")

In [ ]:
def per_benchmark_grid(metric_col, axis_label, suptitle, fname):
    """Small-multiples bar grid: one subplot per benchmark, each a horizontal bar
    chart of the 18 languages' mean `metric_col`, sorted ascending (most efficient
    on top), paradigm-coloured, with a value label on every bar.

    Inputs: the metric column, an axis label (with unit), a figure suptitle, and the
    save_fig basename. Reads df_mean. Side effect: writes a 300-dpi PDF and shows
    the figure. No return value.
    """
    benchmarks = sorted(df_mean['benchmark'].unique())
    ncols = 4
    nrows = int(np.ceil(len(benchmarks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4.8))
    axes = axes.flatten()

    for i, bm in enumerate(benchmarks):
        ax = axes[i]
        sub = (df_mean[df_mean['benchmark'] == bm]
               .set_index('language')[metric_col]
               .sort_values())
        colors = [ps.paradigm_color(l) for l in sub.index]
        bars = ax.barh(sub.index, sub.values, color=colors, alpha=0.85, edgecolor='white')

        x_max = sub.max()
        for bar, val in zip(bars, sub.values):
            lbl = f'{val:,.0f}' if val >= 100 else f'{val:,.1f}'
            ax.text(bar.get_width() + x_max * 0.012, bar.get_y() + bar.get_height() / 2,
                    lbl, va='center', ha='left', fontsize=6, color='#333333')

        ax.set_xlim(0, x_max * 1.20)
        ax.invert_yaxis()                 # most efficient (lowest) at the top
        ax.set_title(bm, fontsize=11)
        ax.set_xlabel(axis_label, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(False)
        ax.grid(axis='x', linestyle='--', alpha=0.4)

    for j in range(i + 1, len(axes)):     # hide any unused panels
        axes[j].set_visible(False)

    legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                      for p in PARADIGM_ORDER]
    fig.legend(handles=legend_handles, title='Paradigm', loc='lower center',
               ncol=3, bbox_to_anchor=(0.5, -0.03))
    fig.suptitle(suptitle, fontsize=15, y=1.01)
    plt.tight_layout()
    ps.save_fig(fig, fname)
    plt.show()

## 1. CPU Energy per Benchmark (J)

In [ ]:
per_benchmark_grid(COL_CPU_ENERGY, 'CPU Energy (J)',
                   'Mean CPU Energy per Benchmark — languages ranked (J)',
                   '05_cpu_energy_per_benchmark')

> **Takeaway:** AOT native compilers lead on every benchmark; the interpreted languages' energy penalty is widest on the compute-bound benchmarks (n-body, spectral-norm, mandelbrot, fannkuch-redux).

## 2. Memory Energy per Benchmark (J)

In [ ]:
per_benchmark_grid(COL_MEM_ENERGY, 'Memory Energy (J)',
                   'Mean Memory (DRAM) Energy per Benchmark — languages ranked (J)',
                   '05_mem_energy_per_benchmark')

> **Takeaway:** DRAM energy is low and fairly uniform on most benchmarks; regex-redux is the outlier, where Erlang's very long run dominates memory energy.

## 3. Execution Time per Benchmark (ms)

In [ ]:
per_benchmark_grid(COL_TIME, 'Execution Time (ms)',
                   'Mean Execution Time per Benchmark — languages ranked (ms)',
                   '05_time_per_benchmark')

> **Takeaway:** the speed ordering is stable across benchmarks (AOT fastest, interpreted slowest); regex-redux and k-nucleotide are the most time-expensive workloads overall.